# BYOD — structural data template (`apsg`)

**Pour in your own field measurements, run all cells, collect stereonets and statistics
in `output/`.**

1. Edit **only** the `CONFIG` cell below.
2. *Run → Run All Cells*.
3. Look at `output/<PROJECT>_*.pdf|xlsx`.

As shipped, the config points to the workshop example files, so the notebook runs as-is.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from apsg import *
from apsg.pandas import pd

## CONFIG — edit this cell

In [ ]:
PROJECT = "field"                                 # used in output file names
FILE = Path("data/field_measurements.csv")        # .csv, .xlsx or .xls
SHEET = 0                                         # Excel sheet (name or number)

# Table layout:
#   "long": one measurement per row, a column says what it is (S0, S1, L1, ...)
#   "wide": one column pair per structure, e.g. S1_azi, S1_inc, L1_azi, L1_inc
LAYOUT = "long"
STRUCTURE_COL = "structure"                       # long layout only
AZI_COL, INC_COL = "azi", "inc"                   # long layout only
WIDE_SUFFIX = ("_azi", "_inc")                    # wide layout only

# How planes were measured: "dd" = dip direction/dip, "rhr" = right-hand-rule strike/dip
PLANE_NOTATION = "dd"

# Which structures are planes and which are lines (names as they appear in the data)
PLANES = ["S0", "S1"]
LINES = ["L1"]

# Folded surface whose pi-girdle gives the fold axis (or None)
FOLDED = "S0"

# Optional fault-slip file with columns fazi, finc, lazi, linc, sense (or None)
FAULTS = Path("data/mele.csv")
FAULT_SENSE_LETTERS = False    # True if the sense column holds letters n/r/s/d instead of 1/-1

## 1. Load

In [ ]:
OUT = Path("output"); OUT.mkdir(exist_ok=True)

def read(path, **kw):
    return pd.read_csv(path) if path.suffix.lower() == ".csv" else pd.read_excel(path, sheet_name=SHEET, **kw)

raw = read(FILE)

def to_long(df):
    # Return a long table with columns structure, azi, inc.
    if LAYOUT == "long":
        out = df.rename(columns={STRUCTURE_COL: "structure", AZI_COL: "azi", INC_COL: "inc"})
        return out[["structure", "azi", "inc"]].dropna()
    parts = []
    for name in PLANES + LINES:
        a, i = name + WIDE_SUFFIX[0], name + WIDE_SUFFIX[1]
        if a in df and i in df:
            parts.append(pd.DataFrame({"structure": name, "azi": df[a], "inc": df[i]}).dropna())
    return pd.concat(parts, ignore_index=True)

data = to_long(raw)
if PLANE_NOTATION == "rhr":                       # strike (RHR) -> dip direction
    is_plane = data["structure"].isin(PLANES)
    data.loc[is_plane, "azi"] = (data.loc[is_plane, "azi"] + 90) % 360
data["structure"].value_counts()

## 2. Build feature sets and statistics

In [ ]:
sets = {}
for name in PLANES:
    sub = data[data["structure"] == name]
    if len(sub):
        sets[name] = sub.fol(name=name)
for name in LINES:
    sub = data[data["structure"] == name]
    if len(sub):
        sets[name] = sub.lin(name=name)

rows = []
for name, fs in sets.items():
    ot = fs.ortensor()
    with np.errstate(over="ignore"):              # harmless overflow for very tight clusters
        st = fs.fisher_statistics()
    rows.append({"structure": name, "type": "plane" if name in PLANES else "line", "n": len(fs),
                 "mean": str(fs.R()), "fisher_k": round(st["k"], 1), "alpha95": round(st["alpha"], 1),
                 "E1": str(ot.eigenlins(0)), "E3": str(ot.eigenlins(2)),
                 "P": round(ot.P, 2), "G": round(ot.G, 2), "R": round(ot.R, 2)})
stats = pd.DataFrame(rows).set_index("structure")
stats

*Reading the table*: `mean`/`alpha95` are meaningful for clustered data (high `P`); for
girdles (high `G`) use the eigenvectors — `E3` of a plane set is the girdle (fold) axis.

## 3. Stereonets

In [ ]:
for name, fs in sets.items():
    s = StereoNet(title=f"{PROJECT}: {name} (n={len(fs)})")
    s.contour(fs, levels=6, alpha=0.7)
    if name in PLANES:
        s.great_circle(fs, color="0.5", lw=0.5)
        s.pole(fs, color="k", ms=3)
    else:
        s.line(fs, color="k", ms=3)
    s.savefig(OUT / f"{PROJECT}_{name}.pdf")
    s.show()

In [ ]:
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
s = StereoNet(title=f"{PROJECT}: all structures")
for (name, fs), c in zip(sets.items(), colors):
    if name in PLANES:
        if fs.ortensor().P > fs.ortensor().G:     # a mean plane only makes sense for a cluster
            s.great_circle(fs.R(), color=c, lw=2, label=f"mean {name}")
        s.pole(fs, color=c, ms=3, label=f"{name} poles")
    else:
        s.line(fs, color=c, ms=3, label=name)
if FOLDED in sets:
    ot = sets[FOLDED].ortensor()
    beta = ot.eigenlins(2)
    s.great_circle(ot.eigenfols(2), color="k", ls="--", label=f"pi-girdle {FOLDED}")
    s.line(beta, color="k", ms=10, label=f"beta {beta}")
    print("fold axis (beta):", beta)
s.savefig(OUT / f"{PROJECT}_summary.pdf")
s.show()

## 4. Faults (optional)

In [ ]:
if FAULTS:
    faults = faultset.from_csv(FAULTS, sense_str=FAULT_SENSE_LETTERS)
    sigma = faults.stress_inversion()
    misfit = faults.angular_misfit(sigma)
    fault_summary = pd.Series({
        "n": len(faults), "sigma1": str(lin(sigma.sigma1dir)), "sigma2": str(lin(sigma.sigma2dir)),
        "sigma3": str(lin(sigma.sigma3dir)), "shape_ratio": round(sigma.shape_ratio, 2),
        "median_misfit": round(float(np.median(misfit)), 1)})
    print(fault_summary.to_string())
    for title, draw in [("Angelier", lambda s: s.fault(faults)), ("Hoeppner", lambda s: s.hoeppner(faults))]:
        s = StereoNet(title=f"{PROJECT} faults: {title}")
        draw(s)
        s.stress(sigma)
        s.savefig(OUT / f"{PROJECT}_faults_{title.lower()}.pdf")
        s.show()

## 5. Export statistics

In [ ]:
xlsx = OUT / f"{PROJECT}_apsg.xlsx"
with pd.ExcelWriter(xlsx) as w:
    stats.to_excel(w, sheet_name="statistics")
    data.to_excel(w, sheet_name="data", index=False)
    if FAULTS:
        fault_summary.to_frame("value").to_excel(w, sheet_name="stress inversion")
        pd.DataFrame({"misfit": misfit}).to_excel(w, sheet_name="fault misfit")
print("written", xlsx)